# Model selection criteria

_Machine Learning — more_

**Reward fit, punish complexity. The best model balances the two.**

A bigger model always fits the training data better. But it may just memorize noise.

     So we need a score that rewards good fit but punishes extra parameters.

---

This notebook is a practice scaffold for the **Model selection criteria** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

rng = np.random.RandomState(0)
x = np.linspace(-3, 3, 60)
y = 2 + 1.5*x - 0.8*x**2 + 0.5*rng.randn(60)   # true degree 2
n = len(y)

print("degree   RSS      AIC      BIC")
best_deg, best_aic = None, np.inf
for deg in range(1, 8):
    Xp = PolynomialFeatures(deg).fit_transform(x.reshape(-1, 1))
    yhat = LinearRegression().fit(Xp, y).predict(Xp)
    rss = float(np.sum((y - yhat)**2))
    k = Xp.shape[1] + 1                          # coefs + noise variance
    lnL = -0.5*n*(np.log(2*np.pi*rss/n) + 1)    # Gaussian log-likelihood
    aic = 2*k - 2*lnL
    bic = k*np.log(n) - 2*lnL
    print("%5d  %7.3f  %7.2f  %7.2f" % (deg, rss, aic, bic))
    if aic < best_aic:
        best_aic, best_deg = aic, deg
print("best degree by AIC = %d" % best_deg)


## Visualize the data & results

_Which model complexity is best on real data?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# REAL data: 569 breast-cancer patients, 30 features.
bc = load_breast_cancer()
X = StandardScaler().fit_transform(bc.data)

ks, cvs = [], []
for k in [1, 3, 5, 7, 9, 15, 25, 45]:
    score = cross_val_score(KNeighborsClassifier(n_neighbors=k),
                            X, bc.target, cv=5).mean()
    ks.append(k); cvs.append(score)      # small k = more complex model

plt.plot(ks, cvs, 'o-', color='#4ea1ff', label='CV accuracy')
plt.xlabel('neighbors k (small k = more complex model)')
plt.ylabel('5-fold cross-validated accuracy')
plt.title('Breast cancer: CV accuracy vs k-NN complexity (peak = best)')
plt.legend(); plt.show()
